# Step 5: Fine-Tuned DistilBERT and an SVM + DistilBERT Ensemble

The F1 experiments showed that changes within the bag-of-words family (separate thresholds, interaction features, a logistic regression ensemble) do not improve the selected Step 2 SVM. The logistic regression made the same errors as the SVM because it used the same features. A fine-tuned transformer reads word order and context, so its errors are more likely to differ from the SVM's, and averaging the two scores could help where averaging two linear models did not.

The earlier DistilBERT pilot trained only the last transformer block on about 60 tokens per posting and was evaluated on a single split. Here all layers are fine-tuned on up to 256 tokens inside the same grouped cross-validation as the SVM. The test set is not used.

**Run this notebook on a GPU** (Colab: *Runtime → Change runtime type → T4 GPU*). One fold takes about 10–15 minutes on a T4. After the run, download the executed notebook (*File → Download → .ipynb*); the result CSV files are also copied to the Drive folder.

### Design (fixed before running)

| | |
|---|---|
| Folds | Repeat 0 of the project protocol (StratifiedGroupKFold, 5 folds, seed 0): the same folds as the saved Step 2 results. |
| Text | `[CLS] title [SEP] company profile [SEP] description [SEP]`, with 24, 64 and 164 tokens per field. |
| DistilBERT | `distilbert-base-uncased` (pinned revision), all layers, 2 epochs, batch 16, AdamW (learning rate 3e-5, weight decay 0.01), linear schedule with 10% warm-up, class-weighted loss, fp16. Not tuned. |
| Threshold | Chosen to maximise F1 on inner out-of-fold scores (3 inner folds, same seeds as the SVM), so DistilBERT is fine-tuned 4 times per fold. |
| Ensemble | Each model's scores are standardised with the mean and standard deviation of its inner out-of-fold scores; the ensemble score is their average. |

### Decision rule

The candidate is the **ensemble**; DistilBERT alone is reported for reference. As in the F1 notebook, the rule is applied to F1, paired against the SVM on the same folds:

1. **5 folds (repeat 0).** Adopt if the mean F1 gain is positive and the ensemble is better in **all 5** folds (one-sided sign test p ≈ 0.031, close to the 12/15 rule's p ≈ 0.018). Reject if the mean gain is not positive or it wins in 3 or fewer folds.
2. **Only if it wins in exactly 4 of 5 folds** with a positive mean gain: run repeats 1 and 2 (10 more folds, `REPEATS = [0, 1, 2]`) and apply the original rule, at least 12 of 15 folds.

## Setup

On Colab this cell clones the repository and installs the pinned scikit-learn version, which is needed to reproduce the grouped split. PyTorch and transformers come with Colab. Results are written to Google Drive after every fold (if `USE_DRIVE = True`), so a disconnected session can resume where it stopped.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("FAKEJOB_SMOKE") == "1"   # tiny local test run, not for results
REPEATS = [0]                                    # [0, 1, 2] only if rule step 2 applies
USE_DRIVE = True

if IN_COLAB:
    if not os.path.exists("SE3513-ML"):
        subprocess.run(["git", "clone", "-q", "https://github.com/fadimedemirbas/SE3513-ML.git"], check=True)
    os.chdir("SE3513-ML")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.6.1", "kagglehub==1.0.3"], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = "/content/drive/MyDrive/se3513_step5"
    else:
        CHECKPOINT_DIR = "results/step5_checkpoints"
else:
    CHECKPOINT_DIR = "results/step5_checkpoints" if not SMOKE else "cache/step5_smoke"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints:", CHECKPOINT_DIR)

In [ ]:
import sklearn

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(f"scikit-learn 1.6.1 is needed to reproduce the split (found {sklearn.__version__}). "
                       "On Colab: Runtime → Restart session, then run all cells again.")

import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from fakejob.data import load_split
from fakejob.evaluation import all_metrics, best_f1_threshold, inner_oof_scores, model_scores
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features
from fakejob.transformer import (CHECKPOINT, REVISION, TrainSettings, encode_postings,
                                 fine_tune_and_score)

transformers.logging.set_verbosity_error()
pd.set_option("display.precision", 3)
METRICS = ["average_precision", "roc_auc", "f1", "precision", "recall"]
GROUP = "empty_company_profile"
SETTINGS = TrainSettings()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda" and not SMOKE:
    raise RuntimeError("No GPU found. On Colab: Runtime → Change runtime type → T4 GPU.")
print(f"Python {sys.version.split()[0]}, torch {torch.__version__}, transformers {transformers.__version__}, "
      f"scikit-learn {sklearn.__version__}, device {torch.cuda.get_device_name() if device.type == 'cuda' else 'cpu'}")

## Data

The assertions check that the grouped split is the project split.

In [ ]:
split = load_split()
features = posting_features(split.model_df)
dev_index = split.dev_index
X_dev = features.loc[dev_index]
y_dev = split.y.loc[dev_index]
groups_dev = split.groups.loc[dev_index]
assert (len(dev_index), int(y_dev.sum())) == (15001, 724)
assert (len(split.test_index), int(split.y.loc[split.test_index].sum())) == (2597, 131)
print(f"Development data: {len(dev_index)} postings, {int(y_dev.sum())} fake")

In [ ]:
if SMOKE:
    # Local test only: the vocabulary of MiniLM is the bert-base-uncased vocabulary, and a tiny
    # randomly initialised DistilBERT replaces the pretrained one.
    from transformers import DistilBertConfig
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    tiny = DistilBertConfig(vocab_size=tokenizer.vocab_size, n_layers=1, n_heads=2, dim=32, hidden_dim=64)
    make_model = lambda: AutoModelForSequenceClassification.from_config(tiny)
else:
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, revision=REVISION)
    make_model = lambda: AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, revision=REVISION, num_labels=2)

input_ids, attention = encode_postings(tokenizer, split.model_df.loc[dev_index])
lengths = attention.sum(dim=1).numpy()
print(f"Tokens per posting: median {np.median(lengths):.0f}, max {lengths.max()}; "
      f"{(lengths >= 240).mean():.1%} of postings use at least 240 of the 256 tokens")

## Cross-Validation

For each outer fold the SVM is refitted with the same inner folds as in Step 2, and its fold metrics are checked against the saved results. DistilBERT is then fine-tuned on each inner training part (for the out-of-fold scores) and on the whole outer training fold (for the test-fold scores).

In [ ]:
saved_svm = pd.read_csv("results/step2_combined_folds.csv").set_index(["repeat", "fold"])
svm_estimator = build_pipeline(STEP2_CONFIG)

def standardise(oof, test):
    mean, std = oof.mean(), oof.std()
    return (oof - mean) / std, (test - mean) / std

def run_fold(repeat, fold, train_pos, test_pos):
    seed = 1000 + repeat * 5 + fold                  # inner seed used by repeated_group_cv
    X_train, y_train, g_train = X_dev.iloc[train_pos], y_dev.iloc[train_pos], groups_dev.iloc[train_pos]

    svm_oof = inner_oof_scores(svm_estimator, X_train, y_train, g_train, 3, seed)
    svm_test = model_scores(clone(svm_estimator).fit(X_train, y_train), X_dev.iloc[test_pos])
    if not SMOKE:  # check before the GPU work that the SVM reproduces the saved Step 2 fold
        threshold = best_f1_threshold(y_train, svm_oof)
        metrics = all_metrics(y_dev.iloc[test_pos], svm_test, threshold)
        saved = saved_svm.loc[(repeat, fold)]
        assert np.allclose([metrics[m] for m in METRICS] + [threshold],
                           [saved[m] for m in METRICS] + [saved["threshold"]]), "SVM does not reproduce Step 2"

    inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    bert_oof = np.empty(len(train_pos))
    for k, (fit_pos, valid_pos) in enumerate(inner.split(X_train, y_train, g_train)):
        fit_rows = train_pos[fit_pos]
        if SMOKE:
            fit_rows = np.random.default_rng(k).choice(fit_rows, 300, replace=False)
        bert_oof[valid_pos] = fine_tune_and_score(make_model, input_ids, attention, y_dev.to_numpy(),
                                                  fit_rows, train_pos[valid_pos], SETTINGS, 100 * seed + k, device)
    fit_rows = train_pos if not SMOKE else np.random.default_rng(9).choice(train_pos, 300, replace=False)
    bert_test = fine_tune_and_score(make_model, input_ids, attention, y_dev.to_numpy(),
                                    fit_rows, test_pos, SETTINGS, 100 * seed + 9, device)

    svm_z_oof, svm_z_test = standardise(svm_oof, svm_test)
    bert_z_oof, bert_z_test = standardise(bert_oof, bert_test)
    return {
        "test_pos": test_pos, "train_pos": train_pos,
        "svm_oof": svm_oof, "svm_test": svm_test, "bert_oof": bert_oof, "bert_test": bert_test,
        "ensemble_oof": (svm_z_oof + bert_z_oof) / 2, "ensemble_test": (svm_z_test + bert_z_test) / 2,
    }

def fold_rows(repeat, fold, scores):
    y_train, y_test = y_dev.to_numpy()[scores["train_pos"]], y_dev.to_numpy()[scores["test_pos"]]
    rows = []
    for model in ["svm", "bert", "ensemble"]:
        threshold = best_f1_threshold(y_train, scores[f"{model}_oof"])
        rows.append({"model": model, "repeat": repeat, "fold": fold, "threshold": threshold,
                     **all_metrics(y_test, scores[f"{model}_test"], threshold)})
    return rows

rows, fold_scores = [], {}
for repeat in REPEATS:
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for fold, (train_pos, test_pos) in enumerate(outer.split(X_dev, y_dev, groups_dev)):
        if SMOKE and fold >= 2:
            break
        path = os.path.join(CHECKPOINT_DIR, f"repeat{repeat}_fold{fold}.npz")
        if os.path.exists(path):
            scores = dict(np.load(path))
            assert np.array_equal(scores["test_pos"], test_pos), "checkpoint from different folds"
            print(f"repeat {repeat} fold {fold}: loaded from checkpoint")
        else:
            start = time.time()
            scores = run_fold(repeat, fold, train_pos, test_pos)
            np.savez(path, **scores)
            print(f"repeat {repeat} fold {fold}: {(time.time() - start) / 60:.1f} min")
        rows += fold_rows(repeat, fold, scores)
        fold_scores[(repeat, fold)] = scores

folds = pd.DataFrame(rows)
print("SVM fold results reproduce Step 2." if not SMOKE else "Smoke run: SVM check skipped.")

## Results

In [ ]:
pivot = folds.pivot_table(index=["repeat", "fold"], columns="model", values=METRICS)
n = len(pivot)
table = []
for model in ["svm", "bert", "ensemble"]:
    row = {"model": model}
    for metric in ["average_precision", "f1", "precision", "recall"]:
        row[metric] = pivot[(metric, model)].mean()
    if model != "svm":
        for metric, label in [("f1", "F1"), ("average_precision", "PR-AUC")]:
            diff = pivot[(metric, model)] - pivot[(metric, "svm")]
            row[f"Δ {label}"] = diff.mean()
            row[f"{label} wins"] = f"{int((diff > 0).sum())}/{n}"
    table.append(row)
overview = pd.DataFrame(table).set_index("model").rename(index={"svm": "SVM (Step 2)", "bert": "DistilBERT", "ensemble": "SVM + DistilBERT"})
display(overview)
display(folds.pivot_table(index=["repeat", "fold"], columns="model", values="f1"))

In [ ]:
f1_diff = pivot[("f1", "ensemble")] - pivot[("f1", "svm")]
wins, gain = int((f1_diff > 0).sum()), f1_diff.mean()
if n == 5:
    if gain > 0 and wins == 5:
        decision = "adopt the ensemble"
    elif gain > 0 and wins == 4:
        decision = "inconclusive: run repeats 1 and 2 (REPEATS = [0, 1, 2]) and apply the 12/15 rule"
    else:
        decision = "keep the SVM"
else:
    decision = "adopt the ensemble" if gain > 0 and wins >= 12 else "keep the SVM"
print(f"Ensemble vs SVM: mean F1 gain {gain:+.4f}, better in {wins}/{n} folds → {decision}")

## Do the Two Models Make Different Errors?

Test-fold scores of all folds (each development posting once per repeat). The correlation of the two score rankings and the errors by profile group show whether DistilBERT adds information the SVM lacks.

In [ ]:
from scipy.stats import spearmanr

records = []
for (repeat, fold), scores in fold_scores.items():
    y_train = y_dev.to_numpy()[scores["train_pos"]]
    part = pd.DataFrame({"repeat": repeat, "fold": fold, "y": y_dev.to_numpy()[scores["test_pos"]],
                         "empty_profile": X_dev[GROUP].to_numpy()[scores["test_pos"]]},
                        index=dev_index[scores["test_pos"]])
    for model in ["svm", "bert", "ensemble"]:
        part[f"{model}_score"] = scores[f"{model}_test"]
        part[f"{model}_flag"] = (scores[f"{model}_test"] >= best_f1_threshold(y_train, scores[f"{model}_oof"])).astype(int)
    records.append(part)
test_scores = pd.concat(records)
first = test_scores[test_scores["repeat"] == REPEATS[0]]
print(f"Spearman correlation of SVM and DistilBERT scores: {spearmanr(first.svm_score, first.bert_score).statistic:.3f}")

fake = first[first.y == 1]
print(f"Fake postings missed by SVM only: {((fake.svm_flag == 0) & (fake.bert_flag == 1)).sum()}, "
      f"by DistilBERT only: {((fake.svm_flag == 1) & (fake.bert_flag == 0)).sum()}, "
      f"by both: {((fake.svm_flag == 0) & (fake.bert_flag == 0)).sum()} (of {len(fake)})")

def errors(frame, model):
    flag = frame[f"{model}_flag"]
    return pd.Series({"missed": int(((frame.y == 1) & (flag == 0)).sum()),
                      "false alarms": int(((frame.y == 0) & (flag == 1)).sum())})

by_profile = pd.concat({model: first.groupby(first.empty_profile.map({0: "filled", 1: "empty"})).apply(errors, model=model)
                        for model in ["svm", "bert", "ensemble"]}, axis=1)
by_profile

In [ ]:
if not SMOKE:
    import shutil
    folds.to_csv("results/step5_distilbert_folds.csv", index=False)
    test_scores.drop(columns=[c for c in test_scores if c.endswith("_flag")]).to_csv(
        "results/step5_distilbert_scores.csv", index_label="row")
    if IN_COLAB:
        # The Colab disk is temporary: keep a copy next to the checkpoints.
        for name in ["step5_distilbert_folds.csv", "step5_distilbert_scores.csv"]:
            shutil.copy(os.path.join("results", name), os.path.join(CHECKPOINT_DIR, name))
    print("Saved results/step5_distilbert_folds.csv and results/step5_distilbert_scores.csv"
          + (f" (copies in {CHECKPOINT_DIR})" if IN_COLAB else ""))

## Findings

FINDINGS_PLACEHOLDER